# Week 6 - Stock KPIs and the Q11 replenishment comparison on the agreed sales scale
**Owners:** Ibrahim Khalil Umar, with Abdibasid Mohamed Ali.

**Why this exists.** Q10 (Week 4) found that purchase receipts are 18.1 times units sold and nothing in the data explains it. The dataset is supplied with the programme, so there is no data owner to ask. The team agreed to use the **sales scale as the working reference** (the stored stock thresholds fit monthly sales) and to label every stock result as resting on that assumption. Week 5 ran Q11 as a labelled scenario. This notebook turns that agreement into the Week 6 result: stock KPIs and the replenishment comparison on the working scale, with the recorded (unscaled) case shown beside it as a sensitivity.

**Every result below rests on the stated assumption:** `WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption`.

**What is simulated.** Daily review of each of the 180 product-branch pairs against its real daily demand (ledger `OUT`), with the pair's mean supplier lead time, lost sales when out of stock, and `opening_stock` as the starting stock. Purchase quantities and `current_stock` are not used for the working-scale results. Lead time is constant, so the results are one historical path, not a forecast.

**Health bar, fixed before looking at the results.** A pair is *at risk* if it fills less than 95% of its demand and *adequate* otherwise. Cover days are shown against the pair's lead time.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
MERGED = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert MERGED is not None, "Could not find the Week 2 merged folder. Run from inside the repository, or set CADETX_MERGED to the folder."
OUT=Path("stock_outputs"); OUT.mkdir(exist_ok=True); CH=Path("stock_charts"); CH.mkdir(exist_ok=True)
LABEL="WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption"
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=.8); ax.set_axisbelow(True)
K=["product_id","branch_id"]
L=pd.read_csv(MERGED/"stock_ledger_enriched.csv",low_memory=False,parse_dates=["movement_date"])
I=pd.read_csv(MERGED/"inventory_with_ledger.csv")
P=pd.read_csv(MERGED/"purchase_orders_full.csv",low_memory=False,usecols=["product_id","branch_id","po_status","lead_time_actual_days"])
S=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,usecols=["so_id","product_id","branch_id","quantity","order_status"])
print("merged data folder found")

merged data folder found


## 1. Gate: is the scale gap still there?

In [2]:
tot=L.groupby("movement_type").quantity.sum(); ratio=tot["IN"]/tot["OUT"]
O=L[L.movement_type=="OUT"]
Dv=S[S.order_status=="Delivered"]
coverage=O.quantity.sum()/Dv.quantity.sum()
gate=pd.DataFrame({"measure":["ledger IN units","ledger OUT units","IN / OUT","ledger OUT as share of delivered sales units","median max_stock","median reorder_level","median safety_stock","median current_stock","pairs with current_stock above max_stock"],
 "value":[int(tot["IN"]),int(tot["OUT"]),round(ratio,2),round(coverage,4),float(I.max_stock.median()),float(I.reorder_level.median()),float(I.safety_stock.median()),float(I.current_stock.median()),int((I.current_stock>I.max_stock).sum())]})
gate["label"]=LABEL; gate.to_csv(OUT/"stock_scale_gate.csv",index=False); print(gate.to_string(index=False))
check("ledger IN is about 18.1 times OUT (Q10 reproduces)","18.13 +/- 0.1",round(ratio,2),abs(ratio-18.13)<0.1)
check("180 product-branch pairs","180",len(I),len(I)==180)
check("ledger OUT holds about 91% of delivered units","0.911 +/- 0.005",round(coverage,4),abs(coverage-0.911)<0.005)
check("current_stock exceeds max_stock for every pair (recorded scale unusable)","180",int((I.current_stock>I.max_stock).sum()),int((I.current_stock>I.max_stock).sum())==180)

                                     measure        value                                                                                 label
                             ledger IN units 2.039780e+07 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                            ledger OUT units 1.124832e+06 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                                    IN / OUT 1.813000e+01 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
ledger OUT as share of delivered sales units 9.108000e-01 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                            median max_stock 3.105000e+02 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                        median reorder_level 6.200000e+01 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this ass

## 2. Inputs: daily demand and lead time per pair
Demand is the ledger `OUT` per pair per day over the full calendar span. Lead time is the mean `lead_time_actual_days` of received purchase-order lines for the pair, rounded (minimum 1 day).

In [3]:
dates=pd.date_range(O.movement_date.min(),O.movement_date.max(),freq="D")
dem=O.groupby(K+["movement_date"]).quantity.sum().unstack("movement_date").reindex(columns=dates,fill_value=0).fillna(0)
dem=dem.reindex(pd.MultiIndex.from_frame(I[K]),fill_value=0)
D=dem.values.astype(float); T=D.shape[1]; years=T/365.25
check("demand matrix total equals ledger OUT total",int(O.quantity.sum()),int(D.sum()),int(D.sum())==int(O.quantity.sum()))
check("demand matrix is 180 pairs by a full daily calendar",f"(180, {len(dates)})",str(D.shape),D.shape==(180,len(dates)))
Pr=P[P.po_status.eq("Received")&P.lead_time_actual_days.notna()]
lt=Pr.groupby(K).lead_time_actual_days.mean().reindex(pd.MultiIndex.from_frame(I[K]))
check("every pair has a lead time from received purchase-order lines","180",int(lt.notna().sum()),bool(lt.notna().all()))
Lead=np.maximum(1,np.round(lt.values)).astype(int)
check("lead times are in a plausible range (5 to 35 days)","5 to 35",f"{Lead.min()} to {Lead.max()}",Lead.min()>=5 and Lead.max()<=35)
mu=D.mean(axis=1); sd=D.std(axis=1,ddof=1)
cost=I.unit_cost.values.astype(float)
print("lead time days: min",Lead.min(),"median",int(np.median(Lead)),"max",Lead.max(),"| span years",round(years,2))
def rop(z): return np.ceil(mu*Lead+z*sd*np.sqrt(Lead))

lead time days: min 17 median 18 max 19 | span years 6.03


## 3. Simulation
Daily review; whenever inventory position (on hand plus on order) is at or below `s`, order up to `S`; the order arrives after the pair's lead time; unmet demand is lost.
- **Policy A (existing):** `s = reorder_level`, `S = max_stock`.
- **Policy B (demand-based trigger):** `s` = mean demand over the lead time plus z times the demand SD times the square root of the lead time (z = 1.28, 1.645, 2.33 for 90%, 95%, 99% service); `S = s + (max_stock - reorder_level)`, so each order keeps the existing size.

In [4]:
def simulate(Drow,open_stock,s,S_,lead):
    n=len(Drow); pipe=np.zeros(n+lead+2); onhand=float(open_stock); on_order=0.0
    unmet=0.0; sdays=0; area=0.0; orders=0; low=0
    for t in range(n):
        onhand+=pipe[t]; on_order-=pipe[t]
        d=Drow[t]; sold=min(d,onhand); unmet+=d-sold
        if d>sold: sdays+=1
        onhand-=sold; pos=onhand+on_order
        if pos<=s:
            q=S_-pos
            if q>0: pipe[t+lead]+=q; on_order+=q; orders+=1
        area+=onhand
    return unmet,sdays,area/n,orders
def run(name,Dm,s_vec,S_vec):
    rows=[]
    for i in range(len(I)):
        u,sd_,avg,n=simulate(Dm[i],I.opening_stock.iloc[i],s_vec[i],S_vec[i],Lead[i])
        tot_d=Dm[i].sum()
        rows.append(dict(product_id=I.product_id.iloc[i],branch_id=I.branch_id.iloc[i],policy=name,s=s_vec[i],S=S_vec[i],total_demand=tot_d,unmet_demand=u,
                         fill_rate=1-u/tot_d,stockout_days=sd_,avg_on_hand=avg,orders_placed=n,avg_stock_value=avg*cost[i]))
    return pd.DataFrame(rows)
s_A=I.reorder_level.values.astype(float); S_A=I.max_stock.values.astype(float)
res=[run("A existing (reorder_level, max_stock)",D,s_A,S_A)]
for z,nm in [(1.28,"B z=1.28 (90%)"),(1.645,"B z=1.645 (95%)"),(2.33,"B z=2.33 (99%)")]:
    sB=rop(z); res.append(run(nm,D,sB,sB+(S_A-s_A)))
R=pd.concat(res,ignore_index=True); R["label"]=LABEL
R.to_csv(OUT/"q11_working_scale_pair_results.csv",index=False)
summ=R.groupby("policy").agg(pairs=("product_id","size"),total_demand=("total_demand","sum"),unmet=("unmet_demand","sum"),stockout_days=("stockout_days","sum"),
    avg_on_hand=("avg_on_hand","mean"),avg_stock_value=("avg_stock_value","sum"),orders=("orders_placed","sum"),
    pairs_below_95=("fill_rate",lambda x:int((x<0.95).sum()))).reset_index()
summ["fill_rate"]=1-summ.unmet/summ.total_demand
summ["share_pair_days_out"]=summ.stockout_days/(len(I)*T)
summ["avg_stock_value_m"]=summ.avg_stock_value/1e6
summ=summ[["policy","fill_rate","share_pair_days_out","avg_on_hand","avg_stock_value_m","orders","pairs_below_95"]]
summ["label"]=LABEL; summ.to_csv(OUT/"q11_working_scale_policy_summary.csv",index=False)
print(summ.round(4).to_string(index=False))

                               policy  fill_rate  share_pair_days_out  avg_on_hand  avg_stock_value_m  orders  pairs_below_95                                                                                 label
A existing (reorder_level, max_stock)     0.9426               0.0140     152.1364           392.5898    4457              72 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                       B z=1.28 (90%)     0.9891               0.0028     170.3423           441.3722    4804               0 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                      B z=1.645 (95%)     0.9939               0.0016     180.2971           468.3394    4839               0 WORKING SCALE - sales scale assumed (Q10 unresolved); results rest on this assumption
                       B z=2.33 (99%)     0.9981               0.0005     199.7031           519.0848    4872               0 WORKING SCALE - sales scal

In [5]:
sm=summ.set_index("policy"); a=sm.iloc[0]
# the Week 5 labelled scenario, same method on the same scale: the rerun must reproduce it
wk5={"fill":[94.26,98.91,99.39,99.81],"orders":[4457,4804,4839,4872],"below":[72,0,0,0]}
check("Policy A fill rate reproduces the Week 5 scenario (94.26%)","94.26",round(a.fill_rate*100,2),abs(a.fill_rate*100-94.26)<0.01)
for i,nm in enumerate(["A","B90","B95","B99"]):
    r=sm.iloc[i]
    check(f"policy {nm}: fill rate, orders and pairs below 95% match Week 5",f"{wk5['fill'][i]}% / {wk5['orders'][i]} / {wk5['below'][i]}",f"{r.fill_rate*100:.2f}% / {int(r.orders)} / {int(r.pairs_below_95)}",
          abs(r.fill_rate*100-wk5['fill'][i])<0.01 and int(r.orders)==wk5['orders'][i] and int(r.pairs_below_95)==wk5['below'][i])
check("every policy keeps the same total demand",f"{int(D.sum())}",f"{int(R.groupby('policy').total_demand.sum().round().nunique())} distinct totals",R.groupby("policy").total_demand.sum().round().nunique()==1)
# independent check: a plain day-by-day re-implementation for five pairs gives the same result
def plain(Drow,open_stock,s,S_,lead):
    onhand=float(open_stock); arrivals={}; unmet=0.0
    for t,d in enumerate(Drow):
        onhand+=arrivals.pop(t,0.0)
        take=min(d,onhand); unmet+=d-take; onhand-=take
        pos=onhand+sum(arrivals.values())
        if pos<=s and S_-pos>0: arrivals[t+lead]=arrivals.get(t+lead,0.0)+(S_-pos)
    return unmet
rowsA=R[R.policy.str.startswith("A")].reset_index(drop=True)
idx=[0,37,90,121,179]
diff=max(abs(plain(D[i],I.opening_stock.iloc[i],s_A[i],S_A[i],Lead[i])-rowsA.unmet_demand.iloc[i]) for i in idx)
check("plain re-implementation matches the simulation on five pairs (max gap in units)","< 0.001",round(float(diff),6),diff<1e-3)

## 4. Stock KPIs on the working scale (existing policy, Policy A)
- **Turnover** = units sold per year divided by average on-hand units (a value version uses unit cost).
- **Days of cover** = average on-hand units divided by mean daily demand.
- **Stock health** = *at risk* if the fill rate is below 95%, otherwise *adequate* (bar fixed above); cover is compared with the lead time.

In [6]:
A=R[R.policy.str.startswith("A")].reset_index(drop=True).copy()
A["mean_daily_demand"]=mu; A["lead_time_days"]=Lead; A["unit_cost"]=cost
A["annual_units_sold"]=A.total_demand/years
A["turnover_per_year"]=A.annual_units_sold/A.avg_on_hand
A["days_of_cover"]=A.avg_on_hand/A.mean_daily_demand
A["cover_minus_lead_time_days"]=A.days_of_cover-A.lead_time_days
A["cover_below_lead_time"]=A.days_of_cover<A.lead_time_days
A["health"]=np.where(A.fill_rate<0.95,"At risk","Adequate")
A["label"]=LABEL
A.to_csv(OUT/"stock_kpis_by_pair.csv",index=False)
cogs=(A.annual_units_sold*A.unit_cost).sum(); avgval=A.avg_stock_value.sum()
port=pd.DataFrame({"measure":["average on-hand units per pair","average stock value at cost (INR M)","annual cost of units sold (INR M)","portfolio turnover per year (cost of units sold / average stock value)","median days of cover","median lead time (days)","pairs with cover below lead time","pairs at risk (fill rate below 95%)","pairs adequate"],
 "value":[round(A.avg_on_hand.mean(),1),round(avgval/1e6,1),round(cogs/1e6,1),round(cogs/avgval,2),round(A.days_of_cover.median(),1),int(np.median(Lead)),int(A.cover_below_lead_time.sum()),int((A.health=="At risk").sum()),int((A.health=="Adequate").sum())]})
port["label"]=LABEL; port.to_csv(OUT/"stock_kpis_portfolio.csv",index=False); print(port.drop(columns="label").to_string(index=False))
bycat=A.merge(I[K+["category"]],on=K).groupby("category").agg(pairs=("product_id","size"),avg_stock_value_m=("avg_stock_value",lambda x:x.sum()/1e6),
     median_turnover=("turnover_per_year","median"),median_cover_days=("days_of_cover","median"),at_risk=("health",lambda x:int((x=="At risk").sum()))).reset_index().round(2)
bycat["label"]=LABEL; bycat.to_csv(OUT/"stock_kpis_by_category.csv",index=False); print(bycat.drop(columns="label").to_string(index=False))
check("stock value of Policy A equals the policy summary","%.1f"%(sm.iloc[0].avg_stock_value_m),"%.1f"%(avgval/1e6),abs(avgval/1e6-sm.iloc[0].avg_stock_value_m)<0.05)
check("pairs at risk equal pairs below 95% fill (health bar applied as set)",str(int(a.pairs_below_95)),str(int((A.health=="At risk").sum())),int(a.pairs_below_95)==int((A.health=="At risk").sum()))
check("turnover recomputed by hand for one pair","%.4f"%(A.annual_units_sold[0]/A.avg_on_hand[0]),"%.4f"%A.turnover_per_year[0],abs(A.annual_units_sold[0]/A.avg_on_hand[0]-A.turnover_per_year[0])<1e-9)
check("category table covers all 180 pairs","180",str(int(bycat.pairs.sum())),int(bycat.pairs.sum())==180)

                                                               measure   value
                                        average on-hand units per pair  152.10
                                   average stock value at cost (INR M)  392.60
                                     annual cost of units sold (INR M) 2718.80
portfolio turnover per year (cost of units sold / average stock value)    6.93
                                                  median days of cover   52.60
                                               median lead time (days)   18.00
                                      pairs with cover below lead time    4.00
                                   pairs at risk (fill rate below 95%)   72.00
                                                        pairs adequate  108.00
     category  pairs  avg_stock_value_m  median_turnover  median_cover_days  at_risk
  Attachments      6               0.39             5.37              68.14        2
        Belts      6               0.33 

## 5. Sensitivity: the recorded (unscaled) case
Two ways to see what happens if the sales scale is *not* the right reference.

1. **Recorded stock levels.** Use `current_stock` as it is stored: turnover and cover days come out absurd, which is why it cannot be used.
2. **Purchase scale taken as true.** Multiply demand by the observed ratio (18.13) and keep the stored thresholds: the existing reorder levels would be far too low.

In [7]:
rec=pd.DataFrame({"product_id":I.product_id,"branch_id":I.branch_id,"current_stock":I.current_stock,"max_stock":I.max_stock,
    "turnover_per_year_recorded":(mu*365.25)/I.current_stock.values,"days_of_cover_recorded":I.current_stock.values/mu})
rec["label"]="SENSITIVITY - recorded current_stock, no scale correction"
rec.to_csv(OUT/"sensitivity_recorded_stock_by_pair.csv",index=False)
Dk=D*ratio
sens=run("A existing at purchase scale (demand x 18.13)",Dk,s_A,S_A)
sens_s=pd.DataFrame({"case":["Working scale (sales), Policy A","Purchase scale (demand x 18.13), Policy A thresholds"],
    "fill_rate":[a.fill_rate,1-sens.unmet_demand.sum()/sens.total_demand.sum()],
    "pairs_below_95":[int(a.pairs_below_95),int((sens.fill_rate<0.95).sum())],
    "median_days_of_cover":[A.days_of_cover.median(),float((sens.avg_on_hand/(mu*ratio)).median())]})
sens_s["label"]="SENSITIVITY"; sens_s.to_csv(OUT/"sensitivity_scale_summary.csv",index=False)
print(f"recorded stock: median turnover {rec.turnover_per_year_recorded.median():.4f} a year, median cover {rec.days_of_cover_recorded.median():,.0f} days")
print(sens_s.round(4).to_string(index=False))
check("recorded cover is more than 10 years of demand for every pair","> 3650 days",f"min {rec.days_of_cover_recorded.min():,.0f}",rec.days_of_cover_recorded.min()>3650)
check("at purchase scale the stored thresholds fill much less than on the working scale","fill < working fill",f"{sens_s.fill_rate[1]:.3f} < {sens_s.fill_rate[0]:.3f}",sens_s.fill_rate[1]<sens_s.fill_rate[0])

recorded stock: median turnover 0.0095 a year, median cover 38,272 days
                                                case  fill_rate  pairs_below_95  median_days_of_cover       label
                     Working scale (sales), Policy A     0.9426              72               52.6014 SENSITIVITY
Purchase scale (demand x 18.13), Policy A thresholds     0.2377             180                1.3679 SENSITIVITY


## 6. Charts

In [8]:
fig,ax=plt.subplots(figsize=(8.6,4.6)); style(ax)
names=["A existing","B 90%","B 95%","B 99%"]; vals=sm.fill_rate.values*100
ax.bar(names,vals,color=[ORANGE,BLUE,BLUE,BLUE],width=.55)
for i,v in enumerate(vals): ax.text(i,v+.15,f"{v:.1f}%",ha="center",color=INK,fontsize=11)
ax.set_ylim(90,100.8); ax.set_ylabel("Share of demand filled, %",color=INK2); ax.set_title("Working scale: fill rate by policy",loc="left",color=INK,fontsize=13)
fig.text(0.01,0.01,LABEL,fontsize=8,color=INK2); plt.tight_layout(rect=(0,0.04,1,1)); plt.savefig(CH/"stock_01_fill_rate_by_policy.png",dpi=150,facecolor=SURF); plt.close()
fig,ax=plt.subplots(figsize=(8.6,4.8)); style(ax)
ax.hist(A.days_of_cover,bins=30,color=BLUE,alpha=.9,edgecolor=SURF)
ax.axvline(np.median(Lead),color=ORANGE,lw=2); ax.text(np.median(Lead)-.6,ax.get_ylim()[1]*.95,"median lead time",color=INK,fontsize=10,ha="right")
ax.set_xlabel("Average days of cover per pair (Policy A)",color=INK2); ax.set_ylabel("Pairs",color=INK2); ax.set_title("Working scale: days of cover against lead time",loc="left",color=INK,fontsize=13)
fig.text(0.01,0.01,LABEL,fontsize=8,color=INK2); plt.tight_layout(rect=(0,0.04,1,1)); plt.savefig(CH/"stock_02_days_of_cover.png",dpi=150,facecolor=SURF); plt.close()
fig,ax=plt.subplots(figsize=(8.6,4.6)); style(ax); ax.set_yscale("log")
vals=[float(A.days_of_cover.median()),float(rec.days_of_cover_recorded.median())]
ax.bar(["Working scale (sales)","Recorded current_stock"],vals,color=[BLUE,ORANGE],width=.5)
for i,v in enumerate(vals): ax.text(i,v*1.15,f"{v:,.0f} days",ha="center",color=INK,fontsize=11)
ax.axhline(np.median(Lead),color=INK2,lw=1,ls="--"); ax.text(1.28,np.median(Lead)*1.12,"median lead time",color=INK2,fontsize=9,ha="right")
ax.set_ylim(5,2e5); ax.set_ylabel("Median days of cover (log scale)",color=INK2); ax.set_title("Recorded stock gives 38,000 days of cover: not usable",loc="left",color=INK,fontsize=13)
fig.text(0.01,0.01,"SENSITIVITY - recorded current_stock, no scale correction",fontsize=8,color=INK2); plt.tight_layout(rect=(0,0.04,1,1)); plt.savefig(CH/"stock_03_recorded_vs_working_cover.png",dpi=150,facecolor=SURF); plt.close()
check("three charts written","3",str(len(list(CH.glob("*.png")))),len(list(CH.glob("*.png")))==3)

## 7. Output files and validation

In [9]:
outs=sorted(p.name for p in OUT.glob("*.csv"))
check("output tables written (eight)","8",str(len(outs)),len(outs)==8)
V=pd.DataFrame(checks); V.insert(0,"Check_ID",range(1,len(V)+1)); V.to_csv("validation_report_week06_stock.csv",index=False)
print(len(V),"checks,",int((V.Status=="PASS").sum()),"pass"); assert (V.Status=="PASS").all()

23 checks, 23 pass
